In [ ]:
#imports
import pandas as pd

In [ ]:
class EnrichOptionsPrices(OptionsMathHelpers, OptionsMathAlgebra):
    
    def __init__(self):
        pass


    def split_iv_tv(self, opt_types, opt_values, spots, strike_pvs):
        opt_values, spots, strike_pvs = self.to_arrays(opt_values, spots, strike_pvs)
        opt_types = self.option_type(opt_types, opt_values)
    
        iv_call, iv_put = self.intrinsic_value(spot=spots, strike_pv=strike_pvs)
        iv = np.where(opt_types == "CALL", iv_call, iv_put)
    
        tv = opt_values - iv

        return iv, tv


    def find_best_tv_by_combo(self, bids_asks, tvs, strikes, expiries):
        tvs, strikes = self.to_arrays(tvs, strikes)
        expiries, bids_asks = self.to_arrays(expiries, bids_asks, dtype=str)
        
        df = pd.DataFrame({
            "tv": tvs,
            "type": bids_asks,
            "strike": strikes,
            "expiry": expiries,
        })

        grouped = df.groupby(["type", "strike", "expiry"])["tv"]

        max_tv = grouped.transform("max")
        min_tv = grouped.transform("min")
    
        best_tv = np.where(bids_asks == "BID", max_tv, min_tv)
        
        return best_tv


    def apply_max_min(self, bids_asks, underlying_prices, tvs, strikes, expiries):
        tvs, strikes = self.to_arrays(tvs, strikes)
        expiries, bids_asks = self.to_arrays(expiries, bids_asks, dtype=str)
    
        unique_expirations = expiries.unique()

        hi_strike_mask = strikes > underlying_prices
        lo_strike_mask = ~hi_strike_mask

        for exp in unique_expirations:
            exp_mask = expiries == exp
        
            # bids first
            joint_mask = lo_strike_mask & exp_mask
            tv_col = df.loc[joint_mask, 'best_tv_strike_bid']
            df.loc[joint_mask, 'enriched_bid_tv'] = tv_col.cummax()
        
            joint_mask = hi_strike_mask & exp_mask
            tv_col = df.loc[joint_mask, 'best_tv_strike_bid']
            df.loc[joint_mask, 'enriched_bid_tv'] = tv_col.iloc[::-1].cummax().iloc[::-1]
            
            # asks second
            joint_mask = hi_strike_mask & exp_mask
            tv_col = df.loc[joint_mask, 'best_tv_strike_ask']
            df.loc[joint_mask, 'enriched_ask_tv'] = tv_col.cummin()
            
            joint_mask = lo_strike_mask & exp_mask
            tv_col = df.loc[joint_mask, 'best_tv_strike_ask']
            df.loc[joint_mask, 'enriched_ask_tv'] = tv_col.iloc[::-1].cummin().iloc[::-1]   






            

    df['enriched_bid_price'] = df['intrinsic'] + df['enriched_bid_tv']
    df['enriched_ask_price'] = df['intrinsic'] + df['enriched_ask_tv']